# Đánh giá YOLO trên BDD100K và so sánh với FastViT

Notebook này dùng các checkpoint `best.pt` trong `experiments/`, chạy inference trên **official BDD100K validation**, rồi đánh giá bằng COCO bbox protocol thống nhất:

- AP (IoU 0.50:0.95), AP50, AP75;
- APS, APM, APL và AR100;
- các metric trên theo từng điều kiện `weather`;
- AP/AP50/AP75 và AP theo kích thước cho từng class;
- biểu đồ và CSV để so sánh YOLO11n, YOLOv10n, YOLO11s, YOLO11x, YOLO26n/s/m/l/x với FastViT.

Prediction được lưu theo COCO JSON, nên có thể chạy lại phần thống kê mà không cần inference lại. Để so sánh FastViT **cùng evaluator và cùng tập ảnh**, đặt `FASTVIT_PREDICTIONS_JSON` tới file `predictions_BDD100K.json` do pipeline FastViT xuất ra; notebook kiểm tra file ground truth đi kèm nếu có. Nếu không cung cấp file này, notebook chỉ dùng hàng tham chiếu tổng thể trong `docs/bdd100k_experiment_report.md`; không suy diễn kết quả FastViT theo weather.

> Lưu ý: evaluator, `maxDets` và image manifest được chuẩn hóa. Tiền xử lý đầu vào và NMS vẫn là implementation riêng của mỗi detector, vì vậy cần giữ cấu hình inference trong artifact khi báo cáo kết quả.

In [ ]:
# Bỏ qua cell này nếu môi trường đã có đủ package.
import subprocess
import sys

INSTALL_DEPENDENCIES = True
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ultralytics", "pycocotools", "pandas", "seaborn"],
        check=True,
    )

## 1. Cấu hình

Các giá trị mặc định bám theo layout Kaggle trong `bdd1000k.ipynb` và layout local của repository. Chỉ cần sửa các biến `*_OVERRIDE` nếu dữ liệu/checkpoint nằm ở nơi khác.

`IMGSZ=800` là độ phân giải đánh giá chung; các run YOLO26 đã train và ghi metric gốc ở 640 nên số tái đánh giá có thể khác `summary_comparison.csv`. `SCORE_THRESHOLD=0.05`, `NMS_IOU=0.5` và `MAX_DETECTIONS=100` khớp cấu hình inference FastViT hiện được ghi trong báo cáo. Nếu thay đổi các giá trị này, hãy đánh giá lại cả FastViT với cùng protocol trước khi kết luận trực tiếp.

In [ ]:
from pathlib import Path

# Đường dẫn tùy chọn. Để None để notebook tự tìm.
REPO_ROOT_OVERRIDE = None
VAL_IMAGE_ROOT_OVERRIDE = None
RAW_VAL_LABELS_OVERRIDE = None
EXPERIMENTS_ROOT_OVERRIDE = None
OUTPUT_ROOT_OVERRIDE = None

# File COCO detections do FastViT export (không phải checkpoint .pth).
# Ví dụ: Path('/mnt/bdd100k/runs/evaluation/json_results/predictions_BDD100K.json')
FASTVIT_PREDICTIONS_JSON = None

# Có thể chỉ định checkpoint thủ công. Giá trị None sẽ dùng đường dẫn mặc định trong experiments/.
MODEL_WEIGHTS_OVERRIDE = {
    "YOLO11n": None,
    "YOLOv10n": None,
    "YOLO11s": None,
    "YOLO11x": None,
    "YOLO26n": None,
    "YOLO26s": None,
    "YOLO26m": None,
    "YOLO26l": None,
    "YOLO26x": None,
}

IMGSZ = 800
BATCH_SIZE = 16
DEVICE = 0                 # 0, 1, ... hoặc 'cpu'
USE_HALF = True            # chỉ áp dụng khi chạy CUDA
SCORE_THRESHOLD = 0.05
NMS_IOU = 0.50
MAX_DETECTIONS = 100
RUN_INFERENCE = True
FORCE_INFERENCE = False    # False: tái sử dụng prediction JSON đã có

# Dùng để smoke test pipeline; phải là None khi lấy số liệu chính thức.
MAX_EVAL_IMAGES = None

CLASSES = [
    "person", "rider", "car", "truck", "bus",
    "train", "motor", "bike", "traffic light", "traffic sign",
]
CLASS_TO_ID = {name: index + 1 for index, name in enumerate(CLASSES)}
CATEGORY_ALIASES = {"pedestrian": "person", "bicycle": "bike", "motorcycle": "motor"}

# Chỉ dùng khi chưa có FASTVIT_PREDICTIONS_JSON. Các số này ở đơn vị %, epoch 4.
FASTVIT_REPORT_REFERENCE = {
    "Model": "FastViT-SA12 (report reference)",
    "AP": 25.66, "AP50": 52.05, "AP75": 21.32,
    "APS": 14.07, "APM": 29.54, "APL": 40.55, "AR100": 39.10,
}

In [ ]:
import contextlib
import hashlib
import io
import json
import os
import platform
import re
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import ultralytics
from IPython.display import display
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm
from ultralytics import YOLO

def first_existing(candidates, description):
    for candidate in candidates:
        if candidate is not None and Path(candidate).expanduser().exists():
            return Path(candidate).expanduser().resolve()
    tried = "\n".join(f"  - {candidate}" for candidate in candidates if candidate is not None)
    raise FileNotFoundError(f"Không tìm thấy {description}. Đã thử:\n{tried}")

def find_repo_root(override=None):
    candidates = [Path(override)] if override is not None else []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents, Path('/kaggle/working/ml-fastvit'), Path('/mnt/ml-fastvit')])
    return first_existing(
        [path for path in candidates if (path / 'train_bdd100k.py').is_file()],
        "repository ml-fastvit",
    )

REPO_ROOT = find_repo_root(REPO_ROOT_OVERRIDE)
EXPERIMENTS_ROOT = Path(EXPERIMENTS_ROOT_OVERRIDE).expanduser().resolve() if EXPERIMENTS_ROOT_OVERRIDE else REPO_ROOT / 'experiments'
default_output_root = Path('/kaggle/working/yolo_bdd100k_evaluation') if Path('/kaggle/working').is_dir() else REPO_ROOT / 'evaluation_outputs' / 'yolo_bdd100k'
OUTPUT_ROOT = Path(OUTPUT_ROOT_OVERRIDE).expanduser().resolve() if OUTPUT_ROOT_OVERRIDE else default_output_root
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

kaggle_root = Path('/kaggle/input/datasets/solesensei/solesensei_bdd100k')
VAL_IMAGE_ROOT = first_existing([
    VAL_IMAGE_ROOT_OVERRIDE,
    kaggle_root / 'bdd100k' / 'bdd100k' / 'images' / '100k' / 'val',
    Path('/mnt/bdd100k/images/100k/val'),
    REPO_ROOT / 'data' / 'bdd100k' / 'images' / '100k' / 'val',
], "thư mục ảnh BDD100K val")
RAW_VAL_LABELS = first_existing([
    RAW_VAL_LABELS_OVERRIDE,
    kaggle_root / 'bdd100k_labels_release' / 'bdd100k' / 'labels' / 'bdd100k_labels_images_val.json',
    Path('/mnt/bdd100k/labels/det_20/det_val.json'),
    Path('/mnt/bdd100k/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json'),
    REPO_ROOT / 'data' / 'bdd100k' / 'labels' / 'det_20' / 'det_val.json',
    REPO_ROOT / 'data' / 'bdd100k' / 'bdd100k_labels_release' / 'bdd100k' / 'labels' / 'bdd100k_labels_images_val.json',
], "BDD100K raw validation labels (cần attributes.weather)")

print(f"REPO_ROOT       : {REPO_ROOT}")
print(f"EXPERIMENTS_ROOT: {EXPERIMENTS_ROOT}")
print(f"VAL_IMAGE_ROOT  : {VAL_IMAGE_ROOT}")
print(f"RAW_VAL_LABELS  : {RAW_VAL_LABELS}")
print(f"OUTPUT_ROOT     : {OUTPUT_ROOT}")
print(f"Python {platform.python_version()} | torch {torch.__version__} | CUDA={torch.cuda.is_available()}")
if MAX_EVAL_IMAGES is not None:
    print("CẢNH BÁO: MAX_EVAL_IMAGES đang bật; kết quả không phải official full validation.")

## 2. Dựng image manifest và COCO ground truth

ID ảnh được gán theo đúng thứ tự frame trong raw BDD100K JSON, giống converter của repository. Notebook kiểm tra ảnh thiếu/trùng basename trước khi inference và giữ cả ảnh không có bounding box trong evaluator.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}

def normalize_weather(value):
    value = str(value or 'undefined').strip().lower().replace(' ', '_')
    return 'undefined' if value in {'', 'unknown'} else value

def build_image_index(root):
    by_name = defaultdict(list)
    for path in sorted(Path(root).rglob('*')):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            by_name[path.name].append(path.resolve())
    duplicates = {name: paths for name, paths in by_name.items() if len(paths) > 1}
    if duplicates:
        example = next(iter(duplicates.items()))
        raise ValueError(f"Có {len(duplicates)} basename ảnh bị trùng; ví dụ: {example}")
    return {name: paths[0] for name, paths in by_name.items()}

with RAW_VAL_LABELS.open('r', encoding='utf-8') as handle:
    raw_frames = json.load(handle)
if not isinstance(raw_frames, list):
    raise TypeError('Raw BDD100K labels phải là một JSON list.')

image_index = build_image_index(VAL_IMAGE_ROOT)
missing_images = [item.get('name') for item in raw_frames if Path(item.get('name', '')).name not in image_index]
if missing_images:
    raise FileNotFoundError(f"Thiếu {len(missing_images)} ảnh validation; ví dụ {missing_images[:5]}")

selected_frames = raw_frames if MAX_EVAL_IMAGES is None else raw_frames[:int(MAX_EVAL_IMAGES)]
coco_images = []
coco_annotations = []
image_paths = []
image_id_to_weather = {}
annotation_id = 1

for image_id, frame in enumerate(selected_frames, start=1):
    image_name = Path(frame['name']).name
    image_path = image_index[image_name]
    weather = normalize_weather(frame.get('attributes', {}).get('weather'))
    # BDD100K detection images trong pipeline này có kích thước gốc 1280x720.
    coco_images.append({
        'id': image_id, 'file_name': image_name, 'width': 1280, 'height': 720, 'weather': weather,
    })
    image_paths.append(image_path)
    image_id_to_weather[image_id] = weather

    for label in frame.get('labels') or []:
        category = CATEGORY_ALIASES.get(label.get('category'), label.get('category'))
        box = label.get('box2d')
        if category not in CLASS_TO_ID or not box:
            continue
        x1 = min(1280.0, max(0.0, float(box['x1'])))
        y1 = min(720.0, max(0.0, float(box['y1'])))
        x2 = min(1280.0, max(0.0, float(box['x2'])))
        y2 = min(720.0, max(0.0, float(box['y2'])))
        width, height = x2 - x1, y2 - y1
        if width <= 0 or height <= 0:
            continue
        coco_annotations.append({
            'id': annotation_id, 'image_id': image_id,
            'category_id': CLASS_TO_ID[category],
            'bbox': [x1, y1, width, height], 'area': width * height, 'iscrowd': 0,
        })
        annotation_id += 1

coco_dataset = {
    'info': {'description': 'BDD100K official validation - 10 detection classes'},
    'licenses': [],
    'images': coco_images,
    'annotations': coco_annotations,
    'categories': [{'id': index + 1, 'name': name} for index, name in enumerate(CLASSES)],
}
ground_truth_path = OUTPUT_ROOT / 'ground_truth_bdd100k_val.json'
ground_truth_path.write_text(json.dumps(coco_dataset), encoding='utf-8')

weather_counts = Counter(image_id_to_weather.values())
print(f"Images: {len(coco_images):,} | boxes: {len(coco_annotations):,} | classes: {len(CLASSES)}")
display(pd.DataFrame(weather_counts.most_common(), columns=['Weather', 'Images']))
print(f"COCO ground truth: {ground_truth_path}")

## 3. Tìm và kiểm tra checkpoint trong `experiments`

Tên thư mục YOLO11s hiện là `yolo11n_800_rect`, vì vậy notebook dùng đường dẫn model rõ ràng thay vì suy model từ tên thư mục run. YOLO26 có thứ tự class khác YOLO11; cell này tạo ánh xạ từ tên class trong từng checkpoint sang category ID thống nhất của FastViT.

In [ ]:
DEFAULT_WEIGHT_PATHS = {
    'YOLO11n': EXPERIMENTS_ROOT / 'output_yolo11n' / 'yolo_bdd100k' / 'yolo11n_800_rect' / 'weights' / 'best.pt',
    'YOLOv10n': EXPERIMENTS_ROOT / 'output_yolov10n' / 'yolo_bdd100k' / 'yolov10n_800_rect' / 'weights' / 'best.pt',
    'YOLO11s': EXPERIMENTS_ROOT / 'output_yolo11s' / 'yolo_bdd100k' / 'yolo11n_800_rect' / 'weights' / 'best.pt',
    'YOLO11x': EXPERIMENTS_ROOT / 'output_yolov11x_800_rect' / 'weights' / 'best.pt',
    'YOLO26n': EXPERIMENTS_ROOT / 'results_yolov26n_40eps' / 'runs' / 'detect' / 'yolo26n' / 'weights' / 'best.pt',
    'YOLO26s': EXPERIMENTS_ROOT / 'results_yolov26s_25eps' / 'runs' / 'detect' / 'yolo26s' / 'weights' / 'best.pt',
    'YOLO26m': EXPERIMENTS_ROOT / 'results_yolov26m_14eps' / 'runs' / 'detect' / 'yolo26m' / 'weights' / 'best.pt',
    'YOLO26l': EXPERIMENTS_ROOT / 'results_yolov26l_8eps' / 'runs' / 'detect' / 'yolo26l' / 'weights' / 'best.pt',
    'YOLO26x': EXPERIMENTS_ROOT / 'results_yolov26x_5eps' / 'runs' / 'detect' / 'yolo26x' / 'weights' / 'best.pt',
}

model_weights = {}
for model_name, default_path in DEFAULT_WEIGHT_PATHS.items():
    configured = MODEL_WEIGHTS_OVERRIDE.get(model_name)
    path = Path(configured).expanduser().resolve() if configured else default_path.resolve()
    if not path.is_file():
        raise FileNotFoundError(f"{model_name}: không tìm thấy checkpoint {path}")
    model_weights[model_name] = path

checkpoint_rows = []
checkpoint_class_to_category = {}
for model_name, weight_path in model_weights.items():
    probe = YOLO(str(weight_path))
    checkpoint_names = {int(index): str(name).strip().lower() for index, name in probe.names.items()}
    if set(checkpoint_names) != set(range(len(CLASSES))) or set(checkpoint_names.values()) != set(CLASSES):
        raise ValueError(
            f"{model_name}: class mapping checkpoint không khớp.\n"
            f"Checkpoint: {checkpoint_names}\nExpected classes: {CLASSES}"
        )
    checkpoint_class_to_category[model_name] = {index: CLASS_TO_ID[name] for index, name in checkpoint_names.items()}
    checkpoint_rows.append({
        'Model': model_name, 'Checkpoint': str(weight_path),
        'Size (MiB)': weight_path.stat().st_size / 1024**2,
        'Class order': ', '.join(checkpoint_names[index] for index in sorted(checkpoint_names)),
    })
    del probe
display(pd.DataFrame(checkpoint_rows).round({'Size (MiB)': 1}))

## 4. Inference YOLO và cache COCO predictions

`boxes.xyxy` của Ultralytics đã được đưa về tọa độ ảnh gốc. Notebook đổi class ID của từng checkpoint sang COCO category ID 1..10 **theo tên class** và lưu `[x, y, width, height]`. Cache được tái sử dụng khi checkpoint, manifest và cấu hình inference không đổi; đặt `FORCE_INFERENCE=True` để chạy lại.

In [ ]:
def slugify(value):
    return re.sub(r'[^a-z0-9]+', '_', value.lower()).strip('_')

def cache_signature(model_name, weight_path):
    weight_stat = weight_path.stat()
    label_stat = RAW_VAL_LABELS.stat()
    payload = {
        'model': model_name, 'weight_path': str(weight_path),
        'weight_size': weight_stat.st_size, 'weight_mtime_ns': weight_stat.st_mtime_ns,
        'raw_labels_path': str(RAW_VAL_LABELS),
        'raw_labels_size': label_stat.st_size, 'raw_labels_mtime_ns': label_stat.st_mtime_ns,
        'images': [str(path) for path in image_paths],
        'class_to_category': checkpoint_class_to_category[model_name],
        'imgsz': IMGSZ, 'batch_size': BATCH_SIZE, 'device': str(DEVICE),
        'half': bool(USE_HALF), 'ultralytics_version': ultralytics.__version__,
        'score_threshold': SCORE_THRESHOLD,
        'nms_iou': NMS_IOU, 'max_detections': MAX_DETECTIONS,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode('utf-8')).hexdigest()

def run_yolo_inference(model_name, weight_path, destination):
    model = YOLO(str(weight_path))
    detections = []
    started = time.perf_counter()
    use_half = bool(USE_HALF and torch.cuda.is_available() and str(DEVICE).lower() != 'cpu')

    for start in tqdm(range(0, len(image_paths), BATCH_SIZE), desc=model_name):
        batch_paths = image_paths[start:start + BATCH_SIZE]
        results = model.predict(
            source=[str(path) for path in batch_paths],
            imgsz=IMGSZ, conf=SCORE_THRESHOLD, iou=NMS_IOU,
            max_det=MAX_DETECTIONS, device=DEVICE, half=use_half,
            augment=False, agnostic_nms=False, verbose=False, stream=False,
        )
        if len(results) != len(batch_paths):
            raise RuntimeError(f"{model_name}: nhận {len(results)} results cho {len(batch_paths)} ảnh")

        for offset, result in enumerate(results):
            image_id = start + offset + 1
            boxes = result.boxes
            if boxes is None or len(boxes) == 0:
                continue
            xyxy = boxes.xyxy.detach().cpu().numpy()
            scores = boxes.conf.detach().cpu().numpy()
            labels = boxes.cls.detach().cpu().numpy().astype(int)
            for box, score, label in zip(xyxy, scores, labels):
                x1, y1, x2, y2 = map(float, box)
                if not 0 <= label < len(CLASSES):
                    raise ValueError(f"Class ID ngoài khoảng: {label}")
                detections.append({
                    'image_id': image_id, 'category_id': checkpoint_class_to_category[model_name][label],
                    'bbox': [x1, y1, max(0.0, x2 - x1), max(0.0, y2 - y1)],
                    'score': float(score),
                })

    temporary_path = destination.with_suffix('.json.tmp')
    temporary_path.write_text(json.dumps(detections), encoding='utf-8')
    temporary_path.replace(destination)
    elapsed = time.perf_counter() - started
    print(f"{model_name}: {len(detections):,} detections | {elapsed / 60:.1f} phút | {destination}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return destination

prediction_files = {}
for model_name, weight_path in model_weights.items():
    destination = OUTPUT_ROOT / f"predictions_{slugify(model_name)}.json"
    cache_metadata_path = destination.with_suffix('.meta.json')
    signature = cache_signature(model_name, weight_path)
    metadata = json.loads(cache_metadata_path.read_text(encoding='utf-8')) if cache_metadata_path.is_file() else {}
    if destination.is_file() and metadata.get('signature') == signature and not FORCE_INFERENCE:
        print(f"{model_name}: dùng cache {destination}")
    elif RUN_INFERENCE:
        run_yolo_inference(model_name, weight_path, destination)
        cache_metadata_path.write_text(json.dumps({'signature': signature}, indent=2), encoding='utf-8')
    else:
        print(f"{model_name}: chưa có cache và RUN_INFERENCE=False; bỏ qua.")
        continue
    prediction_files[model_name] = destination

if FASTVIT_PREDICTIONS_JSON is not None:
    fastvit_path = Path(FASTVIT_PREDICTIONS_JSON).expanduser().resolve()
    if not fastvit_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy FastViT predictions: {fastvit_path}")
    fastvit_gt_path = fastvit_path.parent / 'ground_truths_BDD100K.json'
    if fastvit_gt_path.is_file():
        with fastvit_gt_path.open('r', encoding='utf-8') as handle:
            fastvit_gt = json.load(handle)
        selected_ids = {item['id'] for item in coco_images}
        reference_images = {
            (int(item['id']), int(item['width']), int(item['height']))
            for item in fastvit_gt['images'] if int(item['id']) in selected_ids
        }
        current_images = {(int(item['id']), int(item['width']), int(item['height'])) for item in coco_images}
        def annotation_counter(rows):
            return Counter(
                (int(item['image_id']), int(item['category_id']),
                 tuple(round(float(value), 1) for value in item['bbox']))
                for item in rows if int(item['image_id']) in selected_ids
            )
        if reference_images != current_images or annotation_counter(fastvit_gt['annotations']) != annotation_counter(coco_annotations):
            raise ValueError(
                'FastViT ground truth không khớp image manifest/annotations hiện tại. '
                'Hãy dùng prediction JSON của đúng validation và class mapping này.'
            )
        print(f'FastViT ground truth khớp: {fastvit_gt_path}')
    else:
        print('CẢNH BÁO: thiếu ground_truths_BDD100K.json; chỉ kiểm tra được prediction image IDs.')
    prediction_files['FastViT-SA12'] = fastvit_path

if not prediction_files:
    raise RuntimeError('Không có prediction JSON nào để đánh giá.')

## 5. COCO evaluator: tổng thể, weather, kích thước và class

Protocol dưới đây khớp evaluator FastViT trong repository: 10 IoU thresholds từ 0.50 đến 0.95, 101 recall thresholds, area small `<32²`, medium `32²..96²`, large `>96²`, và `maxDets=[1, 10, 100]`.

In [ ]:
COCO_IOU_THRESHOLDS = np.linspace(0.50, 0.95, 10)
COCO_RECALL_THRESHOLDS = np.linspace(0.0, 1.0, 101)
COCO_AREA_RANGES = [[0.0, 1e10], [0.0, 32.0**2], [32.0**2, 96.0**2], [96.0**2, 1e10]]
COCO_AREA_LABELS = ['all', 'small', 'medium', 'large']
COCO_MAX_DETS = [1, 10, 100]
METRIC_NAMES = ['AP', 'AP50', 'AP75', 'APS', 'APM', 'APL', 'AR1', 'AR10', 'AR100', 'ARS', 'ARM', 'ARL']

def mean_valid(values):
    values = np.asarray(values)
    values = values[values > -1]
    return float(values.mean()) if values.size else np.nan

def validate_predictions(rows, valid_image_ids):
    required = {'image_id', 'category_id', 'bbox', 'score'}
    bad_images = set()
    for index, row in enumerate(rows):
        missing = required - set(row)
        if missing:
            raise ValueError(f"Prediction row {index} thiếu keys {sorted(missing)}")
        if int(row['image_id']) not in valid_image_ids:
            bad_images.add(int(row['image_id']))
        if int(row['category_id']) not in range(1, len(CLASSES) + 1):
            raise ValueError(f"Prediction row {index} có category_id không hợp lệ: {row['category_id']}")
        if len(row['bbox']) != 4 or float(row['bbox'][2]) < 0 or float(row['bbox'][3]) < 0:
            raise ValueError(f"Prediction row {index} có bbox không hợp lệ: {row['bbox']}")
    if bad_images:
        raise ValueError(f"Predictions tham chiếu image IDs không có trong GT: {sorted(bad_images)[:10]}")

def make_coco_eval(coco_gt, coco_dt, image_ids):
    with contextlib.redirect_stdout(io.StringIO()):
        evaluator = COCOeval(coco_gt, coco_dt, iouType='bbox')
        evaluator.params.imgIds = list(image_ids)
        evaluator.params.catIds = list(range(1, len(CLASSES) + 1))
        evaluator.params.iouThrs = COCO_IOU_THRESHOLDS.copy()
        evaluator.params.recThrs = COCO_RECALL_THRESHOLDS.copy()
        evaluator.params.maxDets = COCO_MAX_DETS.copy()
        evaluator.params.areaRng = [values.copy() for values in COCO_AREA_RANGES]
        evaluator.params.areaRngLbl = COCO_AREA_LABELS.copy()
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()
    return evaluator

def aggregate_metrics(evaluator):
    return {
        name: (np.nan if float(value) < 0 else float(value) * 100.0)
        for name, value in zip(METRIC_NAMES, evaluator.stats)
    }

def per_class_metrics(evaluator, model_name):
    precision = evaluator.eval['precision']  # T, R, K, A, M
    rows = []
    iou_50 = int(np.flatnonzero(np.isclose(evaluator.params.iouThrs, 0.50))[0])
    iou_75 = int(np.flatnonzero(np.isclose(evaluator.params.iouThrs, 0.75))[0])
    for class_index, class_name in enumerate(CLASSES):
        rows.append({
            'Model': model_name, 'Class': class_name,
            'AP': mean_valid(precision[:, :, class_index, 0, -1]) * 100,
            'AP50': mean_valid(precision[iou_50, :, class_index, 0, -1]) * 100,
            'AP75': mean_valid(precision[iou_75, :, class_index, 0, -1]) * 100,
            'APS': mean_valid(precision[:, :, class_index, 1, -1]) * 100,
            'APM': mean_valid(precision[:, :, class_index, 2, -1]) * 100,
            'APL': mean_valid(precision[:, :, class_index, 3, -1]) * 100,
        })
    return rows

with contextlib.redirect_stdout(io.StringIO()):
    coco_gt = COCO(str(ground_truth_path))
all_image_ids = [item['id'] for item in coco_images]
valid_image_ids = set(all_image_ids)
weather_image_ids = {
    weather: [image_id for image_id, value in image_id_to_weather.items() if value == weather]
    for weather in sorted(set(image_id_to_weather.values()))
}
annotations_per_image = Counter(item['image_id'] for item in coco_annotations)

overall_rows = []
weather_rows = []
class_rows = []

for model_name, prediction_path in prediction_files.items():
    print(f"Evaluating {model_name}: {prediction_path}")
    with prediction_path.open('r', encoding='utf-8') as handle:
        detections = json.load(handle)
    if model_name == 'FastViT-SA12' and MAX_EVAL_IMAGES is not None:
        detections = [row for row in detections if int(row['image_id']) in valid_image_ids]
    validate_predictions(detections, valid_image_ids)
    if not detections:
        raise ValueError(f'{model_name}: prediction JSON rỗng; không thể chạy COCOeval.loadRes().')
    with contextlib.redirect_stdout(io.StringIO()):
        coco_dt = coco_gt.loadRes(detections)

    full_eval = make_coco_eval(coco_gt, coco_dt, all_image_ids)
    overall_rows.append({'Model': model_name, **aggregate_metrics(full_eval), 'Source': 're-evaluated COCO JSON'})
    class_rows.extend(per_class_metrics(full_eval, model_name))

    for weather, subset_ids in weather_image_ids.items():
        subset_eval = make_coco_eval(coco_gt, coco_dt, subset_ids)
        weather_rows.append({
            'Model': model_name, 'Weather': weather,
            'Images': len(subset_ids),
            'Boxes': sum(annotations_per_image[image_id] for image_id in subset_ids),
            **aggregate_metrics(subset_eval),
        })

overall_df = pd.DataFrame(overall_rows)
weather_df = pd.DataFrame(weather_rows)
class_df = pd.DataFrame(class_rows)
print('Đã hoàn tất COCO evaluation.')

## 6. Bảng so sánh tổng thể với FastViT

Nếu đã cung cấp FastViT prediction JSON thì hàng FastViT dưới đây được tính lại như YOLO. Nếu chưa, hàng `(report reference)` chỉ là mốc từ báo cáo và cột `Source` giúp tránh nhầm nó với kết quả tái đánh giá.

In [ ]:
if 'FastViT-SA12' not in prediction_files:
    reference_row = {**FASTVIT_REPORT_REFERENCE, 'Source': 'docs/bdd100k_experiment_report.md; not re-evaluated'}
    for metric in ['AR1', 'AR10', 'ARS', 'ARM', 'ARL']:
        reference_row[metric] = np.nan
    overall_df = pd.concat([overall_df, pd.DataFrame([reference_row])], ignore_index=True)

overall_columns = ['Model', 'AP', 'AP50', 'AP75', 'APS', 'APM', 'APL', 'AR100', 'Source']
overall_display = overall_df[overall_columns].sort_values('AP', ascending=False).reset_index(drop=True)
display(overall_display.style.format({name: '{:.2f}' for name in overall_columns if name not in {'Model', 'Source'}}, na_rep='—'))

plot_metrics = ['AP', 'AP50', 'AP75', 'APS', 'APM', 'APL']
plot_df = overall_display.melt(id_vars='Model', value_vars=plot_metrics, var_name='Metric', value_name='Score')
plt.figure(figsize=(14, 6))
sns.barplot(data=plot_df, x='Metric', y='Score', hue='Model')
plt.ylabel('Score (%)')
plt.title('BDD100K validation — COCO bbox metrics')
plt.ylim(0, max(65, float(plot_df['Score'].max(skipna=True)) * 1.15))
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'overall_metrics.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. So sánh theo weather

Mỗi hàng dùng toàn bộ ground truth và prediction của đúng nhóm weather đó. APS/APM/APL có thể là `NaN` nếu một subset không có ground truth hợp lệ ở dải kích thước tương ứng. FastViT chỉ xuất hiện khi có prediction JSON.

In [ ]:
weather_columns = ['Model', 'Weather', 'Images', 'Boxes', 'AP', 'AP50', 'AP75', 'APS', 'APM', 'APL', 'AR100']
weather_display = weather_df[weather_columns].sort_values(['Weather', 'AP'], ascending=[True, False]).reset_index(drop=True)
display(weather_display.style.format({name: '{:.2f}' for name in ['AP', 'AP50', 'AP75', 'APS', 'APM', 'APL', 'AR100']}, na_rep='—'))

for metric in ['AP', 'APS', 'APM', 'APL']:
    pivot = weather_df.pivot(index='Weather', columns='Model', values=metric)
    plt.figure(figsize=(max(8, 1.8 * len(pivot.columns)), max(4, 0.65 * len(pivot.index))))
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', vmin=0, cbar_kws={'label': f'{metric} (%)'})
    plt.title(f'BDD100K {metric} theo weather')
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f"weather_{metric.lower()}.png", dpi=180, bbox_inches='tight')
    plt.show()

## 8. So sánh theo class và kích thước vật thể

Bảng chi tiết hữu ích để kiểm tra các class khó của FastViT như `rider`, `motor`, `bike`, hoặc xem một model tốt ở AP tổng nhưng yếu ở vật thể nhỏ. FastViT chỉ xuất hiện khi có prediction JSON.

In [ ]:
class_columns = ['Model', 'Class', 'AP', 'AP50', 'AP75', 'APS', 'APM', 'APL']
display(class_df[class_columns].sort_values(['Class', 'AP'], ascending=[True, False]).style.format(
    {name: '{:.2f}' for name in ['AP', 'AP50', 'AP75', 'APS', 'APM', 'APL']}, na_rep='—'
))

class_ap = class_df.pivot(index='Class', columns='Model', values='AP').reindex(CLASSES)
plt.figure(figsize=(max(9, 1.8 * len(class_ap.columns)), 7))
sns.heatmap(class_ap, annot=True, fmt='.1f', cmap='mako', vmin=0, cbar_kws={'label': 'AP (%)'})
plt.title('AP theo class trên BDD100K validation')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'class_ap.png', dpi=180, bbox_inches='tight')
plt.show()

## 9. Lưu bảng và metadata

Các file CSV dùng đơn vị phần trăm. `evaluation_manifest.json` lưu đường dẫn checkpoint/prediction và toàn bộ tham số inference/evaluation cần thiết để tái lập phép so sánh.

In [ ]:
overall_csv = OUTPUT_ROOT / 'metrics_overall.csv'
weather_csv = OUTPUT_ROOT / 'metrics_by_weather.csv'
class_csv = OUTPUT_ROOT / 'metrics_by_class.csv'
overall_df.to_csv(overall_csv, index=False)
weather_df.to_csv(weather_csv, index=False)
class_df.to_csv(class_csv, index=False)

manifest = {
    'dataset': 'BDD100K official validation',
    'num_images': len(coco_images),
    'num_annotations': len(coco_annotations),
    'raw_labels': str(RAW_VAL_LABELS),
    'image_root': str(VAL_IMAGE_ROOT),
    'ground_truth_json': str(ground_truth_path),
    'weights': {name: str(path) for name, path in model_weights.items()},
    'checkpoint_class_to_category': checkpoint_class_to_category,
    'predictions': {name: str(path) for name, path in prediction_files.items()},
    'inference': {
        'imgsz': IMGSZ, 'batch_size': BATCH_SIZE, 'device': str(DEVICE),
        'half': bool(USE_HALF), 'score_threshold': SCORE_THRESHOLD,
        'nms_iou': NMS_IOU, 'max_detections': MAX_DETECTIONS,
    },
    'coco_protocol': {
        'iou_thresholds': COCO_IOU_THRESHOLDS.tolist(),
        'recall_thresholds': len(COCO_RECALL_THRESHOLDS),
        'area_ranges': COCO_AREA_RANGES, 'area_labels': COCO_AREA_LABELS,
        'max_detections': COCO_MAX_DETS,
    },
    'fastvit_reference_only': 'FastViT-SA12' not in prediction_files,
}
manifest_path = OUTPUT_ROOT / 'evaluation_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

print('Artifacts:')
for path in [overall_csv, weather_csv, class_csv, manifest_path]:
    print(' -', path)

## Cách đọc kết quả

1. Dùng `AP` để xếp hạng chất lượng tổng quát; `AP50` phản ánh khả năng tìm thấy vật thể, còn `AP75` nhạy hơn với độ khít của box.
2. So `APS/APM/APL` để biết lợi thế đến từ kích thước nào. Với BDD100K, `traffic light`, `traffic sign`, `rider`, `motor`, `bike` thường chịu ảnh hưởng mạnh ở dải small.
3. Ở bảng weather, luôn xem cả `Images` và `Boxes`: subset nhỏ như snowy có phương sai cao và không nên được diễn giải ngang với clear/overcast.
4. Chỉ kết luận FastViT theo weather/class khi FastViT đã được nạp từ prediction JSON. Hàng report reference chỉ đủ để đối chiếu các metric tổng thể đã công bố.
5. Khi đưa vào báo cáo, giữ `evaluation_manifest.json` cạnh CSV để lưu vết checkpoint và protocol.